# ILN05 - Intento 3: Ensemble clásico + RoBERTa

## 1. Preparación del entorno

In [ ]:
!unzip Dataset-Movies.zip

In [ ]:
import random
import zipfile

import numpy as np
import pandas as pd

import torch
from torch.utils.data import Dataset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    accuracy_score,
    hamming_loss,
    precision_recall_fscore_support,
    f1_score,
    classification_report
)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline, FeatureUnion

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(42)

## 2. Carga de datos

In [ ]:
train_df = pd.read_csv("dataset_train.csv")
test_df = pd.read_csv("dataset_test.csv")

print("Train:", train_df.shape)
print("Test:", test_df.shape)

display(train_df.head())
display(test_df.head())

## 3. Preparación de etiquetas y texto

In [ ]:
train_df["genre_list"] = train_df["genre"].apply(
    lambda x: [g.strip() for g in x.split(",")]
)

mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(train_df["genre_list"])

genres = list(mlb.classes_)

print("Géneros:", genres)
print("Y shape:", Y.shape)

In [ ]:
train_df["text"] = (
    "Title: " + train_df["movie_name"].fillna("") +
    ". Plot: " + train_df["description"].fillna("")
)

test_df["text"] = (
    "Title: " + test_df["movie_name"].fillna("") +
    ". Plot: " + test_df["description"].fillna("")
)

display(train_df[["movie_name", "genre", "text"]].head())

## 4. División de entrenamiento y validación

In [ ]:
X = train_df["text"].tolist()
X_test = test_df["text"].tolist()

X_train, X_dev, y_train, y_dev = train_test_split(
    X,
    Y,
    test_size=0.2,
    random_state=42
)

print("X_train:", len(X_train))
print("X_dev:", len(X_dev))
print("X_test:", len(X_test))
print("y_train:", y_train.shape)
print("y_dev:", y_dev.shape)

## 5. Métricas y búsqueda de umbrales

In [ ]:
def compute_metrics_official(y_true, y_pred):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )
    acc = accuracy_score(y_true, y_pred)
    hl = hamming_loss(y_true, y_pred)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall,
        "hamming_loss": hl
    }


def compute_metrics_debug(y_true, y_pred):
    metrics = compute_metrics_official(y_true, y_pred)
    metrics["avg_labels_pred"] = y_pred.sum(axis=1).mean()
    metrics["avg_labels_true"] = y_true.sum(axis=1).mean()
    return metrics

In [ ]:
def find_best_thresholds(y_true, y_scores, genres, thresholds=np.arange(0.05, 0.96, 0.01)):
    best_thresholds = []
    best_f1s = []

    for j, genre in enumerate(genres):
        best_t = 0.5
        best_f1 = 0.0

        for t in thresholds:
            y_pred_j = (y_scores[:, j] >= t).astype(int)
            f1 = f1_score(y_true[:, j], y_pred_j, zero_division=0)

            if f1 > best_f1:
                best_f1 = f1
                best_t = t

        best_thresholds.append(best_t)
        best_f1s.append(best_f1)

    return np.array(best_thresholds), np.array(best_f1s)

## 6. Configuración de RoBERTa

In [ ]:
MODEL_NAME = "roberta-base"
MAX_LEN = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
class MovieGenreDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            add_special_tokens=True,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "labels": torch.tensor(self.labels[idx], dtype=torch.float)
        }

In [ ]:
train_dataset = MovieGenreDataset(X_train, y_train, tokenizer, MAX_LEN)
dev_dataset = MovieGenreDataset(X_dev, y_dev, tokenizer, MAX_LEN)
test_dataset = MovieGenreDataset(
    X_test,
    np.zeros((len(X_test), len(genres))),
    tokenizer,
    MAX_LEN
)

print(len(train_dataset), len(dev_dataset), len(test_dataset))

In [ ]:
model_roberta = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(genres),
    problem_type="multi_label_classification"
)

In [ ]:
def compute_metrics_trainer(eval_pred):
    logits, labels = eval_pred

    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)

    return compute_metrics_official(labels, preds)

In [ ]:
training_args = TrainingArguments(
    output_dir="./roberta_base_ensemble_results",
    num_train_epochs=4,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none"
)

In [ ]:
print(training_args.device)

## 7. Entrenamiento y evaluación de RoBERTa

In [ ]:
trainer_roberta = Trainer(
    model=model_roberta,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    compute_metrics=compute_metrics_trainer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print(trainer_roberta.args.device)

trainer_roberta.train()

In [ ]:
eval_roberta = trainer_roberta.evaluate()
display(eval_roberta)

In [ ]:
dev_outputs_roberta = trainer_roberta.predict(dev_dataset)

dev_logits_roberta = dev_outputs_roberta.predictions
dev_probs_roberta = 1 / (1 + np.exp(-dev_logits_roberta))

test_outputs_roberta = trainer_roberta.predict(test_dataset)

test_logits_roberta = test_outputs_roberta.predictions
test_probs_roberta = 1 / (1 + np.exp(-test_logits_roberta))

print("dev_probs_roberta:", dev_probs_roberta.shape)
print("test_probs_roberta:", test_probs_roberta.shape)

In [ ]:
best_thresholds_roberta, best_f1s_roberta = find_best_thresholds(
    y_dev,
    dev_probs_roberta,
    genres
)

y_pred_roberta_thr = (dev_probs_roberta >= best_thresholds_roberta).astype(int)

metrics_roberta_thr = compute_metrics_debug(y_dev, y_pred_roberta_thr)
display(metrics_roberta_thr)

thresholds_roberta_df = pd.DataFrame({
    "genre": genres,
    "threshold": best_thresholds_roberta,
    "dev_f1_class": best_f1s_roberta,
    "support_dev": y_dev.sum(axis=0)
}).sort_values("genre")

display(thresholds_roberta_df)

## 8. Modelo clásico TF-IDF + regresión logística

In [ ]:
classic_model = Pipeline([
    ("features", FeatureUnion([
        ("word_tfidf", TfidfVectorizer(
            lowercase=True,
            strip_accents="unicode",
            analyzer="word",
            ngram_range=(1, 3),
            min_df=2,
            max_df=0.9,
            max_features=50000,
            sublinear_tf=True,
            stop_words="english"
        )),
        ("char_tfidf", TfidfVectorizer(
            lowercase=True,
            strip_accents="unicode",
            analyzer="char_wb",
            ngram_range=(3, 5),
            min_df=2,
            max_features=50000,
            sublinear_tf=True
        ))
    ])),
    ("classifier", OneVsRestClassifier(
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            C=2.0,
            solver="liblinear"
        )
    ))
])

classic_model.fit(X_train, y_train)

dev_probs_classic = classic_model.predict_proba(X_dev)
test_probs_classic = classic_model.predict_proba(X_test)

print("dev_probs_classic:", dev_probs_classic.shape)
print("test_probs_classic:", test_probs_classic.shape)

In [ ]:
best_thresholds_classic, best_f1s_classic = find_best_thresholds(
    y_dev,
    dev_probs_classic,
    genres
)

y_pred_classic_thr = (dev_probs_classic >= best_thresholds_classic).astype(int)

metrics_classic_thr = compute_metrics_debug(y_dev, y_pred_classic_thr)
display(metrics_classic_thr)

## 9. Ensemble de probabilidades

In [ ]:
ensemble_results = []

for alpha in [0.5, 0.6, 0.7, 0.8, 0.9]:
    dev_probs_ensemble = alpha * dev_probs_roberta + (1 - alpha) * dev_probs_classic

    thresholds_ensemble, f1s_ensemble = find_best_thresholds(
        y_dev,
        dev_probs_ensemble,
        genres
    )

    y_pred_ensemble = (dev_probs_ensemble >= thresholds_ensemble).astype(int)

    metrics = compute_metrics_debug(y_dev, y_pred_ensemble)
    metrics["alpha_roberta"] = alpha
    ensemble_results.append(metrics)

ensemble_results_df = pd.DataFrame(ensemble_results).sort_values("f1", ascending=False)
display(ensemble_results_df)

In [ ]:
ensemble_results_fine = []

for alpha in np.arange(0.50, 0.76, 0.025):
    dev_probs_ensemble = alpha * dev_probs_roberta + (1 - alpha) * dev_probs_classic

    thresholds_ensemble, f1s_ensemble = find_best_thresholds(
        y_dev,
        dev_probs_ensemble,
        genres
    )

    y_pred_ensemble = (dev_probs_ensemble >= thresholds_ensemble).astype(int)

    metrics = compute_metrics_debug(y_dev, y_pred_ensemble)
    metrics["alpha_roberta"] = alpha
    ensemble_results_fine.append(metrics)

ensemble_results_fine_df = pd.DataFrame(ensemble_results_fine).sort_values("f1", ascending=False)
display(ensemble_results_fine_df)

In [ ]:
BEST_ALPHA = 0.6

dev_probs_ensemble = BEST_ALPHA * dev_probs_roberta + (1 - BEST_ALPHA) * dev_probs_classic

best_thresholds_ensemble, best_f1s_ensemble = find_best_thresholds(
    y_dev,
    dev_probs_ensemble,
    genres
)

y_pred_ensemble_dev = (dev_probs_ensemble >= best_thresholds_ensemble).astype(int)

metrics_ensemble_dev = compute_metrics_debug(y_dev, y_pred_ensemble_dev)
display(metrics_ensemble_dev)

thresholds_ensemble_df = pd.DataFrame({
    "genre": genres,
    "threshold": best_thresholds_ensemble,
    "dev_f1_class": best_f1s_ensemble,
    "support_dev": y_dev.sum(axis=0)
}).sort_values("genre")

display(thresholds_ensemble_df)

## 10. Predicción sobre test y generación del envío

In [ ]:
test_probs_ensemble = BEST_ALPHA * test_probs_roberta + (1 - BEST_ALPHA) * test_probs_classic

test_pred_bin_ensemble = (test_probs_ensemble >= best_thresholds_ensemble).astype(int)

for i in range(test_pred_bin_ensemble.shape[0]):
    if test_pred_bin_ensemble[i].sum() == 0:
        best_class = test_probs_ensemble[i].argmax()
        test_pred_bin_ensemble[i, best_class] = 1

test_pred_labels_ensemble = mlb.inverse_transform(test_pred_bin_ensemble)

test_pred_genres_ensemble = [
    ", ".join(labels)
    for labels in test_pred_labels_ensemble
]

In [ ]:
submission_ensemble_df = pd.DataFrame({
    "movie_name": test_df["movie_name"],
    "description": test_df["description"],
    "genre": test_pred_genres_ensemble
})

display(submission_ensemble_df.head())
print(submission_ensemble_df.shape)

submission_ensemble_df.to_csv("dataset_test_preds.csv", index=False)

with zipfile.ZipFile("ILN05-Ensemble-TFIDF-RoBERTa.zip", "w") as zipf:
    zipf.write("dataset_test_preds.csv")

with zipfile.ZipFile("ILN05-Ensemble-TFIDF-RoBERTa.zip", "r") as zipf:
    print(zipf.namelist())

## 11. Validación del archivo generado

In [ ]:
check_df = pd.read_csv("dataset_test_preds.csv")

print(check_df.shape)
print(check_df.columns.tolist())
print(check_df["genre"].isna().sum())
print((check_df["genre"].str.len() == 0).sum())
display(check_df.head())